In [0]:
import requests
import time
import json
import os
from datetime import date

base_url = "https://api.company-information.service.gov.uk"

API_KEY = "d6bb76a0-3fdc-4736-b363-23e7e581bc11"

base_path = "/Volumes/investment_intelligence_platform/ingestion/api_raw"

companies = ["00445790", #Tesco PLC
"SC095000", #Lloyds Banking Group PLC
"02723534",#AstraZeneca PLC
"SC045551", #NatWest Group PLC
"04366849",#Shell PLC
"00102498",#J Sainsbury PLC
"02468686",#Aviva PLC
"07524813",#Rolls-Royce Holdings PLC
"04190816",#BT Group PLC
"00293262",#Associated British Foods PLC
"00023307",#Diageo PLC
"03888792",#GLAXOSMITHKLINE PLC
"00617987",#HSBC Holdings PLC
"03236483",#Imperial Brands PLC
"01417162",#Legal & General Group PLC
"01833679",#Vodafone Group Public Limited Company
"04031152",#National Grid PLC
"06270876",#Reckitt Benckiser Group PLC
"00719885",#Rio Tinto PLC
"01397169" #Prudential PLC
] 


endpoints = {
    "company":                  "/company/{company}",
    "officers":                 "/company/{company}/officers",
    "filing-history":           "/company/{company}/filing-history"
}





In [0]:

# --- Date-partitioned folder setup ---
today = date.today()
year, month, day = today.year, today.month, today.day

folders = {
    name: os.path.join(ROOT_PATH, name, str(year), str(month), str(day))
    for name in endpoints
}

for folder in folders.values():
    os.makedirs(folder, exist_ok=True)

#########################################
# --- Fetch helper with error handling ---
def fetch_and_save(url: str, file_path: str, label: str) -> bool:
    try:
        response = requests.get(url, auth=(API_KEY, ""), timeout=10)

        if response.status_code == 200:
            with open(file_path, "w", encoding="utf-8") as f:
                json.dump(response.json(), f, indent=2)
            print(f"  ✓ Saved {label}")
            return True
        else:
            print(f"  ✗ Failed {label} (HTTP {response.status_code})")
            return False

    except requests.exceptions.Timeout:
        print(f"  ✗ Timeout: {label}")
        return False
    except Exception as e:
        print(f"  ✗ Error {label}: {e}")
        return False

#############################
# --- Main ingestion loop ---
##############################
success_count = 0
fail_count = 0

for company in companies:
    company = company.strip()
    print(f"\nProcessing: {company}")

    for endpoint_name, endpoint_template in endpoints.items():
        url = BASE_URL + endpoint_template.format(company=company)
        file_path = os.path.join(folders[endpoint_name], f"{company}.json")
        label = f"{endpoint_name} / {company}"

        ok = fetch_and_save(url, file_path, label)

        if ok:
            success_count += 1
        else:
            fail_count += 1

        time.sleep(0.5)  # Rate limiting — Companies House recommends this

print(f"\n--- Done: {success_count} saved, {fail_count} failed ---")

In [0]:
%pip install yfinance

import yfinance as yf
import pandas as pd
import os
from datetime import datetime

# ---------------------------------------------------
# Tickers (UK equities — London Stock Exchange)
# ---------------------------------------------------
tickers = [
    "TSCO.L",   # Tesco PLC
    "LLOY.L",   # Lloyds Banking Group PLC
    "AZN.L",    # AstraZeneca PLC
    "NWG.L",    # NatWest Group PLC
    "SHEL.L",   # Shell PLC
    "SBRY.L",   # J Sainsbury PLC
    "AV.L",     # Aviva PLC
    "RR.L",     # Rolls-Royce Holdings PLC
    "BT-A.L",   # BT Group PLC
    "ABF.L",    # Associated British Foods PLC
    "DGE.L",    # Diageo PLC
    "GSK.L",    # GlaxoSmithKline PLC
    "HSBA.L",   # HSBC Holdings PLC
    "IMB.L",    # Imperial Brands PLC
    "LGEN.L",   # Legal & General Group PLC
    "VOD.L",    # Vodafone Group PLC
    "NG.L",     # National Grid PLC
    "RKT.L",    # Reckitt Benckiser Group PLC
    "RIO.L",    # Rio Tinto PLC
    "PRU.L",    # Prudential PLC
]
now = datetime.now()
year  = now.strftime("%Y")
month = now.strftime("%m")
day   = now.strftime("%d")

BASE_PATH = "/Volumes/investment_intelligence_platform/ingestion/api_raw/yfinance"

datasets = ["income_statement", "balance_sheet", "cashflow", "history", "stats"]

for d in datasets:
    os.makedirs(f"{BASE_PATH}/{d}/year={year}/month={month}/day={day}", exist_ok=True)


def prepare_financial(df):
    df = df.copy()
    df.columns = [str(c.date()) for c in df.columns]
    df.index.name = "metric"
    return df.reset_index()


def save_dataframe(df: pd.DataFrame, path: str, label: str, orient: str = "records") -> bool:
    try:
        if df is None or df.empty:
            print(f"  ⚠ Empty data: {label}")
            return False
        df.to_json(path, orient=orient)
        print(f"  ✓ Saved {label}")
        return True
    except Exception as e:
        print(f"  ✗ Error {label}: {e}")
        return False


success_count = 0
fail_count = 0

for t in tickers:
    print(f"\nProcessing: {t}")

    try:
        stock = yf.Ticker(t)
    except Exception as e:
        print(f"  ✗ Could not load ticker {t}: {e}")
        fail_count += 5
        continue

    partition = f"year={year}/month={month}/day={day}"

    ok = save_dataframe(
        prepare_financial(stock.financials),
        f"{BASE_PATH}/income_statement/{partition}/{t}.json",
        f"income_statement / {t}"
    )
    success_count += ok; fail_count += not ok

    ok = save_dataframe(
        prepare_financial(stock.balance_sheet),
        f"{BASE_PATH}/balance_sheet/{partition}/{t}.json",
        f"balance_sheet / {t}"
    )
    success_count += ok; fail_count += not ok

    ok = save_dataframe(
        prepare_financial(stock.cashflow),
        f"{BASE_PATH}/cashflow/{partition}/{t}.json",
        f"cashflow / {t}"
    )
    success_count += ok; fail_count += not ok

    ok = save_dataframe(
        stock.history(period="5y"),
        f"{BASE_PATH}/history/{partition}/{t}.json",
        f"history / {t}",
        orient="index"
    )
    success_count += ok; fail_count += not ok

    ok = save_dataframe(
        pd.DataFrame([stock.info]),
        f"{BASE_PATH}/stats/{partition}/{t}.json",
        f"stats / {t}"
    )
    success_count += ok; fail_count += not ok

print(f"\n--- Done: {success_count} saved, {fail_count} failed ---")

In [0]:
import requests
import os

base_url = "https://api.company-information.service.gov.uk"

API_KEY = "d6bb76a0-3fdc-4736-b363-23e7e581bc11"

base_path = "/Volumes/investment_intelligence_platform/ingestion/api_raw"

companies = ["00445790", #Tesco PLC
"SC095000", #Lloyds Banking Group PLC
"02723534",#AstraZeneca PLC
"SC045551", #NatWest Group PLC
"04366849",#Shell PLC
"00102498",#J Sainsbury PLC
"02468686",#Aviva PLC
"07524813",#Rolls-Royce Holdings PLC
"04190816",#BT Group PLC
"00293262",#Associated British Foods PLC
"00023307",#Diageo PLC
"03888792",#GLAXOSMITHKLINE PLC
"00617987",#HSBC Holdings PLC
"03236483",#Imperial Brands PLC
"01417162",#Legal & General Group PLC
"01833679",#Vodafone Group Public Limited Company
"04031152",#National Grid PLC
"06270876",#Reckitt Benckiser Group PLC
"00719885",#Rio Tinto PLC
"01397169" #Prudential PLC
] 



endpoints = [
    "/company/{company}",
    "/company/{company}/registered-office-address",
    "/company/{company}/officers",
    "/company/{company}/charges",
    "/company/{company}/registers",
    "/company/{company}/filing-history"
    
]



In [0]:
for endpoint in endpoints:

    # get clean folder name
    endpoint_name = endpoint.split("/")[-1]

    # fix company endpoint naming
    if endpoint_name == "{company}" or endpoint_name == "company":
        endpoint_name = "company"

    folder_path = f"{base_path}/companies_house/{endpoint_name}"
    os.makedirs(folder_path, exist_ok=True)

    for company in companies:
        company = company.strip()

        url = base_url + endpoint.format(company=company)

        response = requests.get(url, auth=(API_KEY, ""))

        if response.status_code == 200:

            file_path = f"{folder_path}/{company}.json"

            with open(file_path, "w") as f:
                f.write(response.text)

            print(f"Saved {endpoint_name} - {company}")

        else:
            print(f"Failed {endpoint_name} - {company}")

In [0]:
!pip install yfinance
import yfinance as yf
import pandas as pd
import os

In [0]:
!pip install yfinance
import yfinance as yf
import pandas as pd
import os
base_path = "/Volumes/investment_intelligence_platform/ingestion/api_raw/yfinance"

folders = [
    "income_statement",
    "balance_sheet",
    "cashflow",
    "history",
    "stats"
]

for f in folders:
    os.makedirs(f"{base_path}/{f}", exist_ok=True)
dbutils.fs.rm("/Volumes/new_project/raw_data/raw_data/", True)
tickers = ["TSCO.L",
"LLOY.L",
"AZN",
"NWG.L",
"SHEL.L",
"SBRY.L",
"AV.L",
"RR.L",
"BTQ.HM",
"AFO1.F",
"DGE.L",
"GSK.L",
"HSBC",
"IMB.L",
"LGEN.L",
"VOD",
"NGG",
"3RB0.F",
"RIO",
"PRUL.XC"]

for t in tickers:
    print("Processing:", t)

    stock = yf.Ticker(t)

    #  Income Statement (DataFrame)
    income = stock.financials.T
    income.jason(f"{base_path}/income_statement/{t}.jason")

    #  Balance Sheet
    balance = stock.balance_sheet.T
    balance.jason(f"{base_path}/balance_sheet/{t}.jason")

    #  Cash Flow
    cashflow = stock.cashflow.T
    cashflow.jason(f"{base_path}/cashflow/{t}.jason")

    #  1 year history
    history = stock.history(period="5y")
    history.jason(f"{base_path}/history/{t}.jason")

    #  Stats (convert dict → single row DataFrame)
    stats = pd.DataFrame([stock.info])
    stats.jason(f"{base_path}/stats/{t}.jason")

In [0]:
tickers = ["TSCO.L",
"LLOY.L",
"AZN",
"NWG.L",
"SHEL.L",
"SBRY.L",
"AV.L",
"RR.L",
"BTQ.HM",
"AFO1.F",
"DGE.L",
"GSK.L",
"HSBC",
"IMB.L",
"LGEN.L",
"VOD",
"NGG",
"3RB0.F",
"RIO",
"PRUL.XC"]

for t in tickers:
    print("Processing:", t)

    stock = yf.Ticker(t)

    #  Income Statement (DataFrame)
    income = stock.financials.T
    income.jason(f"{base_path}/income_statement/{t}.jason")

    #  Balance Sheet
    balance = stock.balance_sheet.T
    balance.jason(f"{base_path}/balance_sheet/{t}.jason")

    #  Cash Flow
    cashflow = stock.cashflow.T
    cashflow.jason(f"{base_path}/cashflow/{t}.jason")

    #  1 year history
    history = stock.history(period="5y")
    history.jason(f"{base_path}/history/{t}.jason")

    #  Stats (convert dict → single row DataFrame)
    stats = pd.DataFrame([stock.info])
    stats.jason(f"{base_path}/stats/{t}.jason")

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
df=spark.read("/Volumes/investment_intelligence_platform/ingestion/api_raw/bronze/charges/")

In [0]:
df = spark.read.json(
    "/Volumes/investment_intelligence_platform/ingestion/api_raw/bronze/officers/"
)
df.display()